<a href="https://colab.research.google.com/github/pachterlab/cellsweep/blob/main/benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# try:
#     from cellsweep import denoise_count_matrix
# except ImportError:
#     print("cellsweep not found, installing...")
#     !pip install -U -q cellsweep[analysis]

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import itertools
import yaml
import requests
import matplotlib.pyplot as plt
import anndata as ad
from collections import OrderedDict, defaultdict
import seaborn as sns
import scanpy as sc
from IPython.display import Image, display
# from cellbender.remove_background.downstream import anndata_from_h5
from cellsweep.constants import CellBender_Fig2_to_Immune_All_High_celltype_mapping, CellBender_Fig2_to_Immune_All_Low_celltype_mapping, CellTypistHigh_to_ImmuneMajor, CellTypistLow_to_ImmuneMajor
from cellsweep import denoise_count_matrix
from cellsweep.simulation import simulate_cells
import cellsweep.utils as cs_utils
import scipy.sparse as sp

cellsweep_dir = os.path.dirname(os.path.abspath(""))

# Compare technologies

In [ ]:
dataset_name = "simulation1_small_noise"  # options: parse_10x, smartseq_10x
verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
overwrite = True  # overwrite existing files
threads = 8  # for cellsweep and CellBender (if use_cuda=False)

In [ ]:
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name)
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellsweep_dir, "notebooks", "output", dataset_name)
os.makedirs(out_dir, exist_ok=True)

adata_10x_url, adata_parse_url, adata_smartseq_url = None, None, None
adata_path_raw_10x = os.path.join(data_dir, "adata_10x_raw.h5ad")
adata_path_parse = os.path.join(data_dir, "adata_parse_raw.h5ad")
adata_path_smartseq = os.path.join(data_dir, "adata_smartseq_raw.h5ad")

technology_to_adata_raw = {}
if dataset_name == "parse_10x":
    technologies = ["10x", "parse"]
    
    adata_10x_url = ""
    adata_parse_url = ""

    if not os.path.exists(adata_path_raw_10x):
        !wget -O {adata_path_raw_10x} {cfg["adata_10x_url"]}

    if not os.path.exists(adata_path_parse):
        !wget -O {adata_path_parse} {cfg["adata_parse_url"]}
    
    technology_to_adata_raw["10x"] = ad.read_h5ad(adata_path_raw_10x)
    technology_to_adata_raw["parse"] = ad.read_h5ad(adata_path_parse)

    model_pkl = ""

elif dataset_name == "smartseq_10x":
    technologies = ["10x", "smartseq"]
    
    adata_10x_url = ""
    adata_smartseq_url = ""

    if not os.path.exists(adata_path_raw_10x):
        !wget -O {adata_path_raw_10x} {cfg["adata_10x_url"]}
    
    if not os.path.exists(adata_path_smartseq):
        !wget -O {adata_path_smartseq} {cfg["adata_smartseq_url"]}
    
    technology_to_adata_raw["10x"] = ad.read_h5ad(adata_path_raw_10x)
    technology_to_adata_raw["smartseq"] = ad.read_h5ad(adata_path_smartseq)

    model_pkl = ""
else:
    raise ValueError(f"dataset_name {dataset_name} not recognized.")

## Knee plot - use this output to estimate umi_cutoff

In [ ]:
technology_to_umi_cutoff = {}
for technology, adata_raw in technology_to_adata_raw.items():
    print(f"Processing technology: {technology}")
    technology_to_umi_cutoff[technology] = cs_utils.knee_plot(adata_raw, transpose=True, out_path=os.path.join(out_dir, f"knee_plot_{technology}.png"))

In [ ]:
technology_to_umi_cutoff = {}   #!!! update

In [ ]:
for technology, adata_raw in technology_to_adata_raw.items():
    print(f"Inferring empty droplets for technology: {technology}")
    umi_cutoff = technology_to_umi_cutoff[technology]
    adata_raw = cs_utils.infer_empty_droplets(adata_raw, method="threshold", umi_cutoff=umi_cutoff, verbose=verbose)  # adds adata.obs["is_empty"]
    adata_raw.var['empty_counts'] = np.array(adata_raw.X[adata_raw.obs['is_empty'].values, :].sum(axis=0)).flatten()
    technology_to_adata_raw[technology] = adata_raw

## cellsweep

In [ ]:
technology_to_adata_cellsweep = {}
for technology, adata_raw in technology_to_adata_raw.items():
    print(f"Denoising counts for technology: {technology}")
    adata_path_cellsweep = os.path.join(out_dir, f"adata_cellsweep_{technology}.h5ad")
    cellsweep_log_file = os.path.join(out_dir, f"cellsweep_{technology}.log")
    
    if not os.path.exists(adata_path_cellsweep) or overwrite:
        adata = adata_raw.copy()
        if "celltype" not in adata.obs.columns:
            adata = cs_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, verbose=verbose)
        adata_cellsweep = denoise_count_matrix(adata, init_alpha = 0.9, beta = 0.1, adata_out=adata_path_cellsweep, freeze_ambient_profile=True, max_iter=500, empty_droplet_method="threshold", threads=threads, verbose=verbose, log_file=cellsweep_log_file)
    else:
        adata_cellsweep = ad.read_h5ad(adata_path_cellsweep)
    
    adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
    adata_cellsweep.var_names_make_unique()
    
    technology_to_adata_cellsweep[technology] = adata_cellsweep

## Analysis

### Raw vs cellsweep knee plot, scatterplots

In [ ]:
for technology in technologies:
    adata_raw = technology_to_adata_raw[technology]
    adata_cellsweep = technology_to_adata_cellsweep[technology]
    cs_utils.plot_knee_multi([adata_raw, adata_cellsweep], labels=["raw", "cellsweep"], title=f"{technology} Knee Plot", filter_empty=True, transpose=True, out_path=os.path.join(out_dir, f"{technology}_knee_plot.png"))

    cs_utils.plot_matrix_scatterplot(adata_raw, adata_cellsweep, point_type="matrix", density_type="scatter_with_density", scale="log", x_axis="raw", y_axis="cellsweep", out_path=os.path.join(out_dir, f"{technology}_matrix_expression_scatterplot.png"), show=True)
    cs_utils.plot_matrix_scatterplot(adata_raw, adata_cellsweep, point_type="cell", density_type="scatter_with_kde", scale="log", x_axis="raw", y_axis="cellsweep", out_path=os.path.join(out_dir, f"{technology}_cell_expression_scatterplot.png"), show=True)
    cs_utils.plot_matrix_scatterplot(adata_raw, adata_cellsweep, point_type="gene", density_type="scatter_with_kde", scale="log", x_axis="raw", y_axis="cellsweep", out_path=os.path.join(out_dir, f"{technology}_gene_expression_scatterplot.png"), show=True)

### Ambient hat scatterplot across technologies

In [ ]:
technology0, technology1 = technologies[0], technologies[1]
cs_utils.plot_matrix_scatterplot(technology_to_adata_cellsweep[technology0].var["ambient_hat"], technology_to_adata_cellsweep[technology1].var["ambient_hat"], point_type="custom", density_type="scatter_with_kde", scale="log", x_axis=technology0, y_axis=technology1, out_path=os.path.join(out_dir, f"{technology0}_vs_{technology1}_ambient_hat_scatterplot.png"), show=True)